# Folium — praktyczny przewodnik do tworzenia map w Pythonie

Notebook krok po kroku: od najprostszej mapy, przez punkty i formatowanie
warunkowe, po choropleth na warstwie wektorowej i integrację z **Polars**.

Kod celowo jest w większości pisany **bezpośrednio** (bez owijania w funkcje) —
łatwiej wtedy widzieć, co dokładnie się dzieje i które parametry są potrzebne,
a które nie. Funkcje pojawiają się tylko tam, gdzie naprawdę oszczędzają
powtarzania tego samego kodu (sekcja o Polars).

**Instalacja:**
```
pip install folium geopandas polars branca pyarrow
```

In [1]:
import folium
import polars as pl
import pandas as pd
import geopandas as gpd
import branca.colormap as bcm
from folium.plugins import MarkerCluster, HeatMap, Fullscreen, MiniMap, MeasureControl, Draw

## 1. Najprostsza możliwa mapa

To jest absolutne minimum — dwa parametry, bez których mapa nie ma sensu.

In [2]:
m = folium.Map(location=[52.0, 19.0], zoom_start=6)  # location=[lat, lon], zoom_start=poziom przybliżenia
m

### Które parametry `folium.Map()` są faktycznie potrzebne?

| Parametr | Wymagany? | Domyślna wartość | Komentarz |
|---|---|---|---|
| `location` | **praktycznie tak** | środek świata (0,0) | bez tego mapa startuje w oceanie koło Afryki |
| `zoom_start` | **praktycznie tak** | 10 | bez ustawienia zwykle za blisko/za daleko |
| `tiles` | nie | `"OpenStreetMap"` | zmień tylko jeśli chcesz inny podkład |
| `control_scale` | nie | `False` | warto dać `True` — pokazuje skalę w km |
| `zoom_control` | nie | `True` | wyłącz tylko jeśli robisz mapę "statyczną" do prezentacji |
| `width`, `height` | nie | `"100%"` | zmieniaj tylko przy osadzaniu w innym layoucie (np. Streamlit) |

Poniżej ten sam kod z wszystkimi najczęściej używanymi parametrami — reszty
(`min_zoom`, `max_bounds`, `crs` itd.) w praktyce prawie nigdy nie ruszasz.

In [3]:
m = folium.Map(
    location=[52.0, 19.0],   # WYMAGANE w praktyce — [lat, lon] środka widoku
    zoom_start=6,             # WYMAGANE w praktyce — 0 (świat) ... 18+ (budynek)
    tiles="OpenStreetMap",    # opcjonalne — i tak jest domyślne
    control_scale=True,       # opcjonalne, ale polecane — pokazuje skalę
)
m

## 2. Wybór obszaru — `location`/`zoom_start` vs `fit_bounds`

Ręczne wpisywanie `location`/`zoom_start` sprawdza się, gdy znasz obszar na
pamięć (np. zawsze Polska). Gdy obszar zależy od danych (inny za każdym
razem), lepiej dopasować widok automatycznie przez `fit_bounds`.

In [4]:
# fit_bounds przyjmuje [[lat_min, lon_min], [lat_max, lon_max]]
m = folium.Map()  # location/zoom_start można pominąć — fit_bounds i tak je nadpisze

bounds = [[49.0, 14.1], [54.9, 24.2]]  # przybliżony bounding box Polski
m.fit_bounds(bounds)
m

Jeśli masz dane w GeoPandas, bounding box wyciągasz bezpośrednio z warstwy —
`gdf.total_bounds` zwraca `[minx, miny, maxx, maxy]` (uwaga na kolejność:
x=lon, y=lat, czyli odwrotnie niż w `fit_bounds`).

```python
minx, miny, maxx, maxy = gdf.total_bounds
m.fit_bounds([[miny, minx], [maxy, maxx]])
```

**Wymaganie:** `gdf` musi być w `EPSG:4326` (lat/lon) — jeśli nie jest:
`gdf = gdf.to_crs(epsg=4326)`.

## 3. Podkłady mapowe (tiles)

Wbudowany i w pełni darmowy bez klucza API: **`"OpenStreetMap"`**.

⚠️ CartoDB (`cartodbpositron` itd.) i Stamen wymagają teraz własnego klucza
API (Carto / Stadia Maps) — bez niego dostaniesz warning i pustą mapę.
Jeśli chcesz inny motyw, potrzebujesz klucza i customowego URL-a (patrz
komórka niżej).

In [5]:
m = folium.Map(location=[52.0, 19.0], zoom_start=6)

# kilka przełączalnych podkładów — tylko OpenStreetMap nie wymaga klucza
folium.TileLayer("OpenStreetMap", name="OpenStreetMap").add_to(m)

# przykład podkładu z własnym kluczem API (odkomentuj i podstaw swój klucz):
# folium.TileLayer(
#     tiles="https://{s}.basemaps.cartocdn.com/rastertiles/voyager/{z}/{x}/{y}{r}.png?api_key=TWOJ_KLUCZ",
#     attr='&copy; <a href="https://carto.com/attributions">CARTO</a>',
#     name="CartoDB Voyager",
# ).add_to(m)

folium.LayerControl().add_to(m)  # potrzebne TYLKO gdy masz >1 podkład/warstwę do przełączania
m

## 4. Dodawanie punktów — `Marker` vs `CircleMarker`

- **`Marker`** — klasyczna "pinezka", cięższa (obrazek ikony).
- **`CircleMarker`** — kropka SVG, lżejsza i łatwiejsza do stylowania
  kolorem/rozmiarem (dlatego lepsza pod formatowanie warunkowe, sekcja 6).

In [6]:
m = folium.Map(location=[52.23, 21.01], zoom_start=6)

folium.Marker(
    location=[52.23, 21.01],  # WYMAGANE — [lat, lon]
    popup="Warszawa",          # opcjonalne — tekst po kliknięciu
    tooltip="Najedź tutaj",    # opcjonalne — tekst po najechaniu
    icon=folium.Icon(color="blue", icon="info-sign"),  # opcjonalne — domyślnie zwykła niebieska pinezka
).add_to(m)

folium.CircleMarker(
    location=[50.06, 19.94],  # WYMAGANE
    radius=8,                  # opcjonalne, domyślnie 10 — promień w pikselach
    color="#3186cc",           # opcjonalne — kolor obrysu
    fill=True,                 # opcjonalne, domyślnie False — czy wypełnić kółko
    fill_color="#3186cc",      # opcjonalne — działa tylko gdy fill=True
    fill_opacity=0.7,          # opcjonalne, domyślnie 0.2
    weight=1,                  # opcjonalne — grubość obrysu w px
    tooltip="Kraków",
).add_to(m)

m

## 5. Punkty z danych — Polars zamiast pętli po liście

`folium` nie zna Polars, więc dane trzeba "rozpakować" wiersz po wierszu.
Odpowiednikiem `df.iterrows()` z Pandas jest `df.iter_rows(named=True)` —
zwraca dict per wiersz, bez narzutu budowania obiektu Series (szybsze).

In [7]:
points = pl.DataFrame({
    "miasto": ["Warszawa", "Kraków", "Gdańsk", "Wrocław", "Poznań"],
    "lat": [52.23, 50.06, 54.35, 51.11, 52.41],
    "lon": [21.01, 19.94, 18.65, 17.03, 16.93],
    "sprzedaz": [120, 87, 45, 63, 29],
})

m = folium.Map(location=[52.0, 19.0], zoom_start=6)

for row in points.iter_rows(named=True):
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=6,
        popup=f"{row['miasto']}: {row['sprzedaz']}",
        fill=True,
    ).add_to(m)

m

## 6. Warunkowe formatowanie punktów

Zwykły `if/elif` w pętli po wierszach Polars — bez owijania w funkcję,
bo logika jest prosta i czytelniejsza "na wprost".

In [8]:
m = folium.Map(location=[52.0, 19.0], zoom_start=6)

for row in points.iter_rows(named=True):
    wartosc = row["sprzedaz"]

    if wartosc >= 100:
        color = "#1a9641"   # zielony — wysoka sprzedaż
    elif wartosc >= 50:
        color = "#ffbf00"   # żółty — średnia
    else:
        color = "#d7191c"   # czerwony — niska

    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=6 + wartosc / 15,          # promień rośnie z wartością (kodowanie "bąbelkowe")
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.8,
        tooltip=f"{row['miasto']}: {wartosc}",
    ).add_to(m)

m

## 7. Etykiety — `tooltip` (hover) vs `popup` (klik)

- `tooltip` — krótki tekst pokazujący się po najechaniu myszką.
- `popup` — pełny HTML pokazujący się po kliknięciu, zostaje "przypięty".

In [9]:
m = folium.Map(location=[52.23, 21.01], zoom_start=10)

row = points.row(0, named=True)  # pierwszy wiersz jako dict — Warszawa

html = f"""
<b>{row['miasto']}</b><br>
Sprzedaż: {row['sprzedaz']}<br>
Współrzędne: {row['lat']}, {row['lon']}
"""

folium.Marker(
    location=[row["lat"], row["lon"]],
    tooltip=row["miasto"],                                   # krótkie, hover
    popup=folium.Popup(folium.IFrame(html, width=200, height=100), max_width=250),  # rozbudowane, klik
).add_to(m)

m

## 8. Klastrowanie punktów (`MarkerCluster`)

Przy większej liczbie punktów (setki/tysiące) pojedyncze markery zaczynają
zapychać mapę i spowalniać przeglądarkę. `MarkerCluster` grupuje bliskie
punkty w "bąble" z liczbą elementów, rozpadające się po przybliżeniu.

In [10]:
m = folium.Map(location=[52.0, 19.0], zoom_start=6)

cluster = MarkerCluster(name="Punkty (klaster)").add_to(m)

for row in points.iter_rows(named=True):
    folium.Marker(
        location=[row["lat"], row["lon"]],
        popup=row["miasto"],
    ).add_to(cluster)  # UWAGA: dodajemy do klastra, nie bezpośrednio do `m`

m

## 9. Warstwy jako filtr — `FeatureGroup` + `LayerControl`

Każda kategoria trafia do osobnej grupy, którą można włączać/wyłączać
checkboxem — to jest praktyczny odpowiednik "filtra" sterowanego z Pythona,
bez dodatkowego JS.

In [11]:
categorized = pl.DataFrame({
    "miasto": ["Warszawa", "Kraków", "Gdańsk", "Wrocław"],
    "lat": [52.23, 50.06, 54.35, 51.11],
    "lon": [21.01, 19.94, 18.65, 17.03],
    "region": ["Centrum", "Południe", "Północ", "Południe"],
})

m = folium.Map(location=[52.0, 19.0], zoom_start=6)

kolory_regionow = {"Centrum": "blue", "Południe": "green", "Północ": "red"}

for region in categorized["region"].unique().to_list():
    grupa = folium.FeatureGroup(name=region)  # nazwa = etykieta w LayerControl

    subset = categorized.filter(pl.col("region") == region)
    for row in subset.iter_rows(named=True):
        folium.CircleMarker(
            location=[row["lat"], row["lon"]],
            radius=7,
            color=kolory_regionow[region],
            fill=True,
            fill_color=kolory_regionow[region],
            popup=row["miasto"],
        ).add_to(grupa)

    grupa.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)  # WYMAGANE, żeby filtr był w ogóle widoczny
m

## 10. Choropleth na warstwie wektorowej (np. granice województw)

Polars nie obsługuje geometrii — dane atrybutowe (liczby do pokolorowania)
trzymasz w Polars, geometrię w GeoPandas, i łączysz je przez `.to_pandas()`.

Poniżej przykład na syntetycznych "województwach" (3 kwadraty), żeby kod
dało się uruchomić bez pliku zewnętrznego — w praktyce podmień
`gpd.read_file(...)` na swój plik (shapefile/GeoJSON) z granicami.

In [12]:
from shapely.geometry import Polygon

# w praktyce: gdf = gpd.read_file("wojewodztwa.geojson").to_crs(epsg=4326)
gdf = gpd.GeoDataFrame({
    "woj_nazwa": ["A", "B", "C"],
    "geometry": [
        Polygon([(19, 51), (20, 51), (20, 52), (19, 52)]),
        Polygon([(20, 51), (21, 51), (21, 52), (20, 52)]),
        Polygon([(21, 51), (22, 51), (22, 52), (21, 52)]),
    ],
}, crs="EPSG:4326")

wskazniki = pl.DataFrame({"woj_nazwa": ["A", "B", "C"], "wskaznik": [12, 55, 91]})

# merge: geometria (GeoPandas/Pandas) + dane (Polars -> Pandas)
gdf = gdf.merge(wskazniki.to_pandas(), on="woj_nazwa", how="left")

colormap = bcm.linear.YlOrRd_09.scale(gdf["wskaznik"].min(), gdf["wskaznik"].max())
colormap.caption = "Wskaźnik"

def style_function(feature):
    wartosc = feature["properties"]["wskaznik"]
    return {
        "fillColor": colormap(wartosc) if wartosc is not None else "lightgrey",
        "color": "black",       # kolor obrysu
        "weight": 1,
        "fillOpacity": 0.75,
    }

m = folium.Map(location=[51.5, 20.5], zoom_start=8)

folium.GeoJson(
    gdf.__geo_interface__,
    style_function=style_function,
    tooltip=folium.GeoJsonTooltip(fields=["woj_nazwa", "wskaznik"], aliases=["Województwo", "Wskaźnik"]),
).add_to(m)

colormap.add_to(m)  # legenda w rogu mapy
m

> `folium.Choropleth(...)` to szybszy "gotowiec" na to samo, ale z dużo
> mniejszą kontrolą (prostszy tooltip, brak hover-highlight). Wariant z
> `GeoJson` + `style_function` powyżej polecam jako domyślny — łatwiej go
> potem rozbudować.

## 11. HeatMap — mapa gęstości

Tańsza renderingowo alternatywa dla tysięcy pojedynczych punktów, gdy liczy
się koncentracja, a nie pojedyncza wartość.

In [13]:
m = folium.Map(location=[52.0, 19.0], zoom_start=6)

# HeatMap chce listy list [lat, lon] (opcjonalnie [lat, lon, waga])
heat_data = points.select(["lat", "lon"]).rows()  # lista tupli -> ok, folium akceptuje

HeatMap(heat_data, radius=25).add_to(m)
m

## 12. Dodatkowe narzędzia (pluginy)

In [14]:
m = folium.Map(location=[52.0, 19.0], zoom_start=6)

Fullscreen().add_to(m)                              # przycisk pełnego ekranu
MiniMap(toggle_display=True).add_to(m)               # podgląd całości w rogu
MeasureControl(primary_length_unit="kilometers").add_to(m)  # narzędzie do mierzenia odległości/pola
Draw(export=True).add_to(m)                           # rysowanie kształtów przez użytkownika

m

## 13. Zapis mapy

In [15]:
m.save("mapa.html")  # jedyny wymagany parametr — ścieżka pliku wyjściowego

## 14. Integracja z Polars — dwie funkcje, które faktycznie warto mieć

W przeciwieństwie do reszty notebooka, tu dwie rzeczy **opłaca się** owinąć
w funkcję — bo używa się ich wielokrotnie, a logika (konwersja
geometrii / agregacja przestrzenna) jest na tyle nietrywialna, że lepiej
napisać ją raz.

### a) Konwersja Polars → GeoDataFrame

Polars nie ma typu geometrycznego, więc żeby narysować punkty jako warstwę
wektorową (np. do dalszej analizy w GeoPandas/QGIS, nie tylko do folium),
trzeba zbudować geometrię z kolumn lat/lon.

In [16]:
def polars_to_geodataframe(
    df: pl.DataFrame,
    lat_col: str = "lat",
    lon_col: str = "lon",
    crs: str = "EPSG:4326",
) -> gpd.GeoDataFrame:
    """Konwertuje Polars DataFrame z kolumnami lat/lon na GeoDataFrame punktowy."""
    pdf = df.to_pandas()
    geometry = gpd.points_from_xy(pdf[lon_col], pdf[lat_col])  # UWAGA: kolejność (x=lon, y=lat)
    return gpd.GeoDataFrame(pdf, geometry=geometry, crs=crs)


gdf_points = polars_to_geodataframe(points)
gdf_points.head()

,miasto,lat,lon,sprzedaz,geometry
0,Warszawa,52.23,21.01,120,POINT (21.01 52.23)
1,Kraków,50.06,19.94,87,POINT (19.94 50.06)
2,Gdańsk,54.35,18.65,45,POINT (18.65 54.35)
3,Wrocław,51.11,17.03,63,POINT (17.03 51.11)
4,Poznań,52.41,16.93,29,POINT (16.93 52.41)


### b) Agregacja przestrzenna w Polars przed wizualizacją (spatial binning)

Przy dużych zbiorach (dziesiątki/setki tysięcy punktów) rysowanie każdego z
osobna nie ma sensu — szybciej i czytelniej jest zagregować je do siatki
(np. 0.05°) i pokazać sumę/liczbę na komórkę. Polars (silnik w Rust) robi to
zauważalnie szybciej niż Pandas przy większych wolumenach, więc warto
agregować właśnie tutaj, zanim dane w ogóle trafią do folium.

In [17]:
def aggregate_points_to_grid(
    df: pl.DataFrame,
    lat_col: str = "lat",
    lon_col: str = "lon",
    cell_size: float = 0.5,
    value_col: str | None = None,
) -> pl.DataFrame:
    """
    Agreguje punkty do siatki przestrzennej o rozmiarze `cell_size` (w stopniach).
    Zwraca środek każdej komórki + liczbę punktów (`n`) i sumę `value_col` (jeśli podano).
    Przydatne jako krok przed HeatMap albo przed rysowaniem dużego zbioru punktów.
    """
    grid = df.with_columns([
        (pl.col(lat_col) / cell_size).floor().alias("_grid_lat"),
        (pl.col(lon_col) / cell_size).floor().alias("_grid_lon"),
    ])

    agg_exprs = [pl.len().alias("n")]
    if value_col is not None:
        agg_exprs.append(pl.col(value_col).sum().alias(f"{value_col}_suma"))

    result = (
        grid.group_by(["_grid_lat", "_grid_lon"])
        .agg(agg_exprs)
        .with_columns([
            ((pl.col("_grid_lat") + 0.5) * cell_size).alias("lat_center"),
            ((pl.col("_grid_lon") + 0.5) * cell_size).alias("lon_center"),
        ])
        .drop(["_grid_lat", "_grid_lon"])
    )
    return result


zagregowane = aggregate_points_to_grid(points, cell_size=1.0, value_col="sprzedaz")
zagregowane

n,sprzedaz_suma,lat_center,lon_center
u32,i64,f64,f64
1,29,52.5,16.5
1,120,52.5,21.5
1,87,50.5,19.5
1,45,54.5,18.5
1,63,51.5,17.5


In [18]:
# użycie wyniku agregacji jako HeatMap ważonej sumą sprzedaży
m = folium.Map(location=[52.0, 19.0], zoom_start=6)

heat_data = zagregowane.select(["lat_center", "lon_center", "sprzedaz_suma"]).rows()
HeatMap(heat_data, radius=30).add_to(m)
m

## Podsumowanie

- Do prostych, jednorazowych map — kod bezpośredni jak w tym notebooku jest
  najczytelniejszy.
- Do rzeczy używanych w wielu projektach (konwersje, agregacje, powtarzalne
  choroplety) sensowniej trzymać funkcje w osobnym module (`folium_toolkit.py`
  z poprzedniej wiadomości) i importować je tutaj.
- Polars warto włączyć do pipeline'u głównie **przed** etapem rysowania —
  jako szybszy silnik do filtrowania/agregacji dużych zbiorów, zanim dane
  trafią do folium (które i tak renderuje po stronie przeglądarki i nie
  udźwignie surowych milionów punktów).